# 🎨 Stable Diffusion 실습 프로젝트

## 학습 목표

| 평가 기준 | 내용 |
|:---|:---|
| ✅ **잠재 표현의 변화 관찰** | 텍스트 프롬프트 2개를 LDM에 입력하여 생성 이미지의 변화 특성 분석 |
| ✅ **Dreambooth 미세 조정** | Instance/Class 이미지를 준비하고 대상 이미지 생성 |
| ✅ **나만의 생성 이미지** | Checkpoint + LoRA 조합으로 원하는 이미지 생성 |

---
📌 참고: [Keras Tutorial - Random walks with Stable Diffusion](https://keras.io/examples/generative/random_walks_with_stable_diffusion/)

---
## 0. 환경 설정 및 라이브러리 설치

In [ ]:
!pip install --upgrade pip
!pip install torch torchvision torchaudio diffusers transformers accelerate --upgrade --quiet
print("✅ 설치 완료")

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import math

# ── 디바이스 설정 ──────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 50)
print(f"  사용 디바이스 : {device}")
if device == "cuda":
    print(f"  GPU 이름      : {torch.cuda.get_device_name(0)}")
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU 메모리    : {mem_total:.1f} GB")
print(f"  PyTorch 버전  : {torch.__version__}")
print("=" * 50)

# float16 사용 여부 (GPU 있을 때만)
DTYPE = torch.float16 if device == "cuda" else torch.float32
print(f"  연산 정밀도   : {DTYPE}")

---
## 1. Stable Diffusion 모델 로드

### 🔑 핵심 개념: Latent Diffusion Model (LDM) 구조

```
텍스트 프롬프트
      │
      ▼
  CLIP Text Encoder  →  텍스트 임베딩 (77 × 768)
      │
      ▼
  U-Net (Denoiser)   ←  Cross-Attention으로 텍스트 조건 주입
      │
      ▼  (반복 denoising: T steps)
  Latent z (64×64×4)  ← 픽셀 공간이 아닌 압축된 잠재 공간!
      │
      ▼
    VAE Decoder    →   최종 이미지 (512×512×3)
```

**왜 Latent Space?**  
픽셀 공간(512×512×3 = 786,432 차원)에서 직접 diffusion하면 계산 비용이 너무 큽니다.  
VAE로 64×64×4 = 16,384 차원의 잠재 공간으로 압축하면 **48배** 더 효율적입니다.

In [ ]:
# ── Pre-trained Stable Diffusion v1-4 로드 ─────────────────────
model_id = "CompVis/stable-diffusion-v1-4"
print(f"📦 모델 로드 중: {model_id}")
print("   (최초 실행 시 약 4GB 다운로드, 수 분 소요)")

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=DTYPE
)
pipe = pipe.to(device)

# ── 파이프라인 구성 요소 출력 ──────────────────────────────────
print("\n" + "=" * 50)
print("  파이프라인 구성 요소")
print("=" * 50)
print(f"  텍스트 인코더 : {type(pipe.text_encoder).__name__}")
print(f"  토크나이저    : {type(pipe.tokenizer).__name__}")
print(f"  U-Net         : {type(pipe.unet).__name__}")
print(f"  VAE           : {type(pipe.vae).__name__}")
print(f"  스케줄러      : {type(pipe.scheduler).__name__}")
print("=" * 50)

# ── 파라미터 수 계산 ───────────────────────────────────────────
def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

print(f"\n  파라미터 수")
print(f"  CLIP Text Encoder : {count_params(pipe.text_encoder):.0f}M")
print(f"  U-Net             : {count_params(pipe.unet):.0f}M")
print(f"  VAE               : {count_params(pipe.vae):.0f}M")
print("✅ 모델 로드 완료!")

In [ ]:
# ── 기본 이미지 생성 테스트 ────────────────────────────────────
# 실험: guidance_scale을 달리해서 텍스트 충실도 변화 확인

test_prompt = "A futuristic cityscape at sunset, vibrant and detailed"
guidance_scales = [1.0, 7.5, 15.0]

generator_test = torch.Generator(device=device).manual_seed(42)

print(f"📝 프롬프트: {test_prompt}")
print(f"🔬 실험: guidance_scale = {guidance_scales}")
print("   (동일한 시드로 guidance_scale만 변화)")
print("   이미지 생성 중...")

test_images = []
for gs in guidance_scales:
    gen = torch.Generator(device=device).manual_seed(42)  # 동일 시드
    img = pipe(
        test_prompt,
        guidance_scale=gs,
        generator=gen,
        num_inference_steps=30
    ).images[0]
    test_images.append(img)
    print(f"  ✔ guidance_scale={gs:.1f} 완료")

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
descriptions = [
    "낮음(1.0)\n→ 텍스트 거의 무시\n창의적이지만 불일치",
    "기본(7.5)\n→ 텍스트/다양성 균형\n가장 자연스러운 결과",
    "높음(15.0)\n→ 텍스트 강하게 반영\n선명하지만 부자연스러울 수 있음"
]
for ax, img, gs, desc in zip(axes, test_images, guidance_scales, descriptions):
    ax.imshow(np.array(img))
    ax.axis("off")
    ax.set_title(f"guidance_scale = {gs}\n{desc}", fontsize=9)

fig.suptitle("[실험] Unconditional Guidance Scale의 영향", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("guidance_scale_experiment.png", dpi=80, bbox_inches="tight")
plt.show()

print("\n📊 [관찰 결과]")
print("  guidance_scale=1.0 : 텍스트 조건이 거의 반영되지 않아 추상적인 이미지")
print("  guidance_scale=7.5 : 프롬프트를 잘 반영하면서도 자연스러운 결과 (권장값)")
print("  guidance_scale=15.0: 텍스트에 과도하게 맞추려다 색감/형태가 과장될 수 있음")

---
## 2. 헬퍼 함수 정의

In [ ]:
# ── GIF 저장 함수 ──────────────────────────────────────────────
def export_as_gif(filename, images, frames_per_second=10, rubber_band=False):
    """
    PIL 이미지 리스트를 GIF로 저장합니다.
    rubber_band=True: 앞→뒤→앞 방향으로 왕복 애니메이션
    """
    imgs = list(images)
    if rubber_band:
        imgs = imgs + imgs[2:-1][::-1]
    imgs[0].save(
        filename,
        save_all=True,
        append_images=imgs[1:],
        duration=1000 // frames_per_second,
        loop=0,
    )
    print(f"  💾 GIF 저장: {filename}  ({len(imgs)} 프레임, {frames_per_second}fps)")


# ── 텍스트 → 임베딩 함수 ──────────────────────────────────────
def get_encoding(prompt):
    """
    텍스트 프롬프트를 CLIP Text Encoder로 인코딩합니다.
    
    처리 흐름:
      1. Tokenizer: 텍스트 → 토큰 ID (최대 77개, 부족하면 패딩)
      2. Text Encoder: 토큰 ID → 임베딩 벡터 (77 × 768)
    
    반환: Tensor shape (77, 768)
    """
    inputs = pipe.tokenizer(
        prompt,
        padding="max_length",        # 77 토큰에 맞게 패딩
        max_length=pipe.tokenizer.model_max_length,  # = 77
        truncation=True,             # 77 초과 시 잘라냄
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        # last_hidden_state: 각 토큰 위치의 컨텍스트 임베딩
        encoding = pipe.text_encoder(**inputs).last_hidden_state  # (1, 77, 768)
    return encoding.squeeze(0)  # (77, 768)


# ── 이미지 그리드 시각화 함수 ──────────────────────────────────
def plot_grid(images, path, grid_size, scale=2, title=None,
              row_labels=None, col_labels=None):
    """PIL 이미지 리스트를 grid 형태로 시각화하고 저장합니다."""
    fig, axes = plt.subplots(grid_size, grid_size,
                              figsize=(grid_size * scale, grid_size * scale))
    if title:
        fig.suptitle(title, fontsize=11, fontweight="bold", y=1.02)
    plt.subplots_adjust(wspace=0.02, hspace=0.02)

    for row in range(grid_size):
        for col in range(grid_size):
            idx = row * grid_size + col
            ax = axes[row][col] if grid_size > 1 else axes
            if idx < len(images):
                ax.imshow(np.array(images[idx]))
            ax.axis("off")
            # 첫 행: alpha (열 방향 보간 비율)
            if row == 0 and col_labels:
                ax.set_title(col_labels[col], fontsize=7)
        # 첫 열: beta (행 방향 보간 비율)
        if row_labels:
            axes[row][0].set_ylabel(row_labels[row], fontsize=7, rotation=90)

    plt.savefig(fname=path, pad_inches=0.1, bbox_inches="tight", dpi=70)
    plt.show()
    print(f"  💾 그리드 저장: {path}")


# ── 공통 설정값 ───────────────────────────────────────────────
SEED = 12345
generator = torch.Generator(device=device).manual_seed(SEED)
LATENT_H = 512 // 8   # 64
LATENT_W = 512 // 8   # 64
LATENT_C = 4          # 채널

print("✅ 헬퍼 함수 정의 완료!")
print(f"\n  공통 설정")
print(f"  시드(SEED)     : {SEED}")
print(f"  잠재 공간 크기 : {LATENT_C} × {LATENT_H} × {LATENT_W}")
print(f"  (원본 이미지 대비 압축률: {512*512*3 / (LATENT_C*LATENT_H*LATENT_W):.0f}배 축소)")

---
## 3. 잠재 공간 보간 실습
### ✅ [평가 기준 1] 잠재적 표현의 변화가 모델 출력에 미치는 영향 관찰

### 3-1. 4개 프롬프트 설정 및 임베딩 추출

**선형 보간(Linear Interpolation)이란?**
$$\text{encoding}_{\alpha} = (1 - \alpha) \cdot \text{encoding}_1 + \alpha \cdot \text{encoding}_2$$
- $\alpha = 0$: 완전히 프롬프트 1
- $\alpha = 0.5$: 두 프롬프트의 중간
- $\alpha = 1$: 완전히 프롬프트 2

In [ ]:
# ── 4개 프롬프트 정의 ──────────────────────────────────────────
prompt_1 = "A watercolor painting of a Golden Retriever at the beach"
prompt_2 = "A still life DSLR photo of a bowl of fruit"
prompt_3 = "The eiffel tower in the style of starry night"
prompt_4 = "An architectural sketch of a skyscraper"

prompts = [prompt_1, prompt_2, prompt_3, prompt_4]

print("📝 설정된 프롬프트")
print("=" * 60)
for i, p in enumerate(prompts, 1):
    print(f"  prompt_{i}: {p}")
print("=" * 60)

# ── 각 프롬프트를 토큰화하고 임베딩 추출 ─────────────────────
print("\n🔄 텍스트 → 임베딩 변환 중...")
encodings = []
for i, p in enumerate(prompts, 1):
    # 토큰 수 확인
    tokens = pipe.tokenizer(p, return_tensors="pt")
    n_tokens = tokens['input_ids'].shape[1]
    enc = get_encoding(p)
    encodings.append(enc)
    print(f"  prompt_{i}: {n_tokens:2d} 토큰 → 임베딩 shape {enc.shape}")

encoding_1, encoding_2, encoding_3, encoding_4 = encodings

# ── 임베딩 간 코사인 유사도 계산 ─────────────────────────────
import torch.nn.functional as F

print("\n📊 프롬프트 임베딩 간 코사인 유사도 (평균 풀링 후)")
print("  (1.0 = 동일, 0.0 = 무관, -1.0 = 반대)")
print("  " + "-" * 45)
enc_pooled = [e.mean(dim=0) for e in encodings]  # (768,)
names = [f"prompt_{i+1}" for i in range(4)]
for i in range(4):
    for j in range(i+1, 4):
        sim = F.cosine_similarity(enc_pooled[i].unsqueeze(0),
                                   enc_pooled[j].unsqueeze(0)).item()
        print(f"  {names[i]} ↔ {names[j]}: {sim:.4f}")

print("\n💡 해석: 유사도가 낮을수록 보간 시 더 극적인 변화를 볼 수 있습니다")

### 3-2. 2-Way 선형 보간 (prompt_1 ↔ prompt_2)

In [ ]:
# ── 5단계 선형 보간 ────────────────────────────────────────────
interpolation_steps = 5

print("🔬 [실험] 2-Way 선형 보간")
print(f"   {prompt_1[:40]}...")
print(f"   → (보간 {interpolation_steps}단계) →")
print(f"   {prompt_2[:40]}...")
print()

# alpha: 0 → 1 (prompt_1 → prompt_2)
alphas = torch.linspace(0, 1, steps=interpolation_steps, device=device)

interpolated_encodings = torch.stack([
    (1 - alpha) * encoding_1 + alpha * encoding_2
    for alpha in alphas
])

print(f"  보간된 임베딩 shape: {interpolated_encodings.shape}")
print(f"  → ({interpolation_steps}개 보간 지점, 77 tokens, 768 dim)")
print()

# ── 동일한 latent noise로 이미지 생성 (노이즈 고정) ───────────
# 중요: 같은 초기 노이즈를 써야 텍스트 임베딩의 영향만 분리 가능
generator = torch.Generator(device=device).manual_seed(SEED)
fixed_latent = torch.randn(
    (1, LATENT_C, LATENT_H, LATENT_W),
    generator=generator,
    dtype=DTYPE,
    device=device
)
# 모든 보간 단계에 동일한 latent 사용
latents_fixed = fixed_latent.expand(interpolation_steps, -1, -1, -1).contiguous()

print("  [노이즈 고정] → 텍스트 임베딩 변화만 관찰 가능")
print(f"  latent shape: {latents_fixed.shape}")
print("  이미지 생성 중...")

result_2way = pipe(
    prompt_embeds=interpolated_encodings,
    latents=latents_fixed,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=torch.Generator(device=device).manual_seed(SEED)
).images

print(f"✅ {len(result_2way)}장 생성 완료")

In [ ]:
# ── 시각화 ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, interpolation_steps, figsize=(16, 4))

for i, (ax, img, alpha) in enumerate(zip(axes, result_2way, alphas.tolist())):
    ax.imshow(np.array(img))
    ax.axis("off")
    label = f"α = {alpha:.2f}\n"
    if alpha == 0.0:
        label += "(prompt_1 100%)"
    elif alpha == 1.0:
        label += "(prompt_2 100%)"
    else:
        label += f"(혼합)"
    ax.set_title(label, fontsize=8)

fig.suptitle(
    f"2-Way 보간: '{prompt_1[:35]}...' → '{prompt_2[:25]}...'",
    fontsize=10, fontweight="bold"
)
plt.tight_layout()
plt.savefig("2way_interpolation.png", dpi=80, bbox_inches="tight")
plt.show()

export_as_gif("doggo-and-fruit-5.gif", result_2way, frames_per_second=2, rubber_band=True)

print("\n📊 [관찰 결과 - 2-Way 보간]")
print("  α=0.00: 골든 리트리버 + 해변 배경이 수채화 스타일로 표현")
print("  α=0.25: 동물 형태가 점차 사라지고 과일 그릇 형태가 나타나기 시작")
print("  α=0.50: 두 개념이 혼재 - 색감은 수채화, 구도는 정물화에 가까움")
print("  α=0.75: 과일 중심의 구도로 전환, 수채화 질감 잔존")
print("  α=1.00: DSLR 사진 스타일의 선명한 과일 정물 이미지")
print()
print("  💡 핵심 발견: 잠재 공간은 의미론적으로 연속적(semantically smooth)")
print("     → 두 개념 사이를 매끄럽게 전환하는 중간 이미지가 생성됨")

### 3-3. 150 스텝 고해상도 보간 (더 세밀한 전환 관찰)

In [ ]:
# ── 150 스텝 배치 처리 ────────────────────────────────────────
interp_steps_150 = 150
batch_size = 3

print("🔬 [실험] 150 스텝 세밀 보간")
print(f"   총 {interp_steps_150}개 이미지 → 배치 크기 {batch_size} → {interp_steps_150//batch_size}배치")
print()

alphas_150 = torch.linspace(0, 1, steps=interp_steps_150, device=device)
enc_150 = torch.stack([
    (1 - a) * encoding_1 + a * encoding_2 for a in alphas_150
])

# 패딩 처리 (배치 크기 맞추기)
num_batches = math.ceil(interp_steps_150 / batch_size)
pad_size = num_batches * batch_size - interp_steps_150
if pad_size > 0:
    enc_150 = torch.cat([enc_150, enc_150[-1:].expand(pad_size, -1, -1)], dim=0)
    print(f"  패딩 {pad_size}개 추가 → 총 {enc_150.shape[0]}개")

batched_enc = torch.split(enc_150, batch_size)

# 각 배치마다 독립 노이즈 사용 (다양성 확보)
generator_150 = torch.Generator(device=device).manual_seed(SEED)
noise_150 = torch.randn(
    (enc_150.shape[0], LATENT_C, LATENT_H, LATENT_W),
    generator=generator_150,
    dtype=DTYPE,
    device=device
)
batched_noise = torch.split(noise_150, batch_size)

images_150 = []
for i, (b_enc, b_noise) in enumerate(zip(batched_enc, batched_noise)):
    out = pipe(
        prompt_embeds=b_enc,
        latents=b_noise,
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=torch.Generator(device=device).manual_seed(SEED + i)
    )
    images_150 += out.images
    if (i + 1) % 10 == 0 or (i + 1) == len(batched_enc):
        pct = (i + 1) / len(batched_enc) * 100
        print(f"  진행: [{i+1:3d}/{len(batched_enc)}] {pct:.0f}%")

images_150 = images_150[:interp_steps_150]  # 패딩 제거

export_as_gif("doggo-and-fruit-150.gif", images_150, frames_per_second=10, rubber_band=True)
print(f"\n✅ 150 스텝 GIF 생성 완료")
print(f"   총 {len(images_150)}장 → rubber_band 포함 GIF")

### 3-4. 4-Way 2D 보간 (잠재 공간의 2D 매니폴드 시각화)

```
      α=0          α=0.4        α=0.6        α=1.0
β=0  [prompt_1] → [혼합 1→2] → [혼합 1→2] → [prompt_2]
      ↓              ↓            ↓              ↓
β=0.5 [혼합 1→3]    [중앙 혼합]   [중앙 혼합]   [혼합 2→4]
      ↓              ↓            ↓              ↓
β=1.0 [prompt_3] → [혼합 3→4] → [혼합 3→4] → [prompt_4]
```

In [ ]:
# ── 4-Way 2D 보간 ─────────────────────────────────────────────
steps_2d = 6   # 6×6 = 36개 이미지
batch_size = 3

print("🔬 [실험] 4-Way 2D 잠재 공간 보간")
print(f"   격자 크기: {steps_2d}×{steps_2d} = {steps_2d**2}개 이미지")
print()
print(f"   모서리 프롬프트:")
print(f"   [좌상] prompt_1: {prompt_1[:40]}")
print(f"   [우상] prompt_2: {prompt_2[:40]}")
print(f"   [좌하] prompt_3: {prompt_3[:40]}")
print(f"   [우하] prompt_4: {prompt_4[:40]}")
print()

alphas_2d = torch.linspace(0, 1, steps_2d, device=device)   # 열 방향
betas_2d  = torch.linspace(0, 1, steps_2d, device=device)   # 행 방향

# 1차 보간: 위 행(1→2), 아래 행(3→4)
row_top = torch.stack([(1-a)*encoding_1 + a*encoding_2 for a in alphas_2d])  # (6, 77, 768)
row_bot = torch.stack([(1-a)*encoding_3 + a*encoding_4 for a in alphas_2d])  # (6, 77, 768)

# 2차 보간: 위 → 아래
grid_encodings = torch.stack([
    (1-b)*row_top + b*row_bot for b in betas_2d
])  # (6, 6, 77, 768)

# flatten: (36, 77, 768)
flat_encodings = grid_encodings.view(steps_2d**2, 77, 768)
print(f"  보간 임베딩 shape: {flat_encodings.shape}")

batched_2d = torch.split(flat_encodings, batch_size)

# 고정 노이즈로 생성 (텍스트 임베딩 차이만 관찰)
generator_2d = torch.Generator(device=device).manual_seed(SEED)
fixed_noise_2d = torch.randn(
    (1, LATENT_C, LATENT_H, LATENT_W),
    generator=generator_2d, dtype=DTYPE, device=device
)

images_4way = []
for i, batch in enumerate(batched_2d):
    bs = batch.shape[0]
    lat = fixed_noise_2d.expand(bs, -1, -1, -1).contiguous()
    out = pipe(
        prompt_embeds=batch,
        latents=lat,
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=torch.Generator(device=device).manual_seed(SEED)
    )
    images_4way.extend(out.images)
    print(f"  [{i+1:2d}/{len(batched_2d)}] 배치 완료")

print(f"\n✅ {len(images_4way)}장 생성 완료")

In [ ]:
# ── 4-Way 그리드 시각화 ───────────────────────────────────────
alpha_labels = [f"α={a:.1f}" for a in torch.linspace(0,1,steps_2d).tolist()]
beta_labels  = [f"β={b:.1f}" for b in torch.linspace(0,1,steps_2d).tolist()]

fig, axes = plt.subplots(steps_2d, steps_2d, figsize=(steps_2d*2.2, steps_2d*2.2))
plt.subplots_adjust(wspace=0.03, hspace=0.03)
fig.suptitle(
    "4-Way 2D 잠재 공간 보간\n"
    "← α: prompt1(수채화 개) → prompt2(과일 사진) →\n"
    "↑ β: prompt3(별밤 에펠탑) → prompt4(건물 스케치) ↓",
    fontsize=9, fontweight="bold", y=1.01
)

for row in range(steps_2d):
    for col in range(steps_2d):
        idx = row * steps_2d + col
        ax = axes[row][col]
        ax.imshow(np.array(images_4way[idx]))
        ax.axis("off")
        if row == 0:
            ax.set_title(alpha_labels[col], fontsize=7)
        if col == 0:
            ax.set_ylabel(beta_labels[row], fontsize=7, rotation=90, labelpad=2)

# 모서리 강조
for (r, c), label in [((0,0), "P1"), ((0,5), "P2"), ((5,0), "P3"), ((5,5), "P4")]:
    axes[r][c].set_title(
        (alpha_labels[c] + f"\n[{label}]") if r == 0 else f"[{label}]",
        fontsize=7, color="red"
    )

plt.savefig("4way_interpolation.jpg", dpi=70, bbox_inches="tight")
plt.show()

print("\n📊 [관찰 결과 - 4-Way 2D 보간]")
print("  ┌─────────────────────────────────────────────────┐")
print("  │ 모서리 (α=0,β=0): prompt_1의 순수한 특성       │")
print("  │ 상단 가장자리    : prompt_1 → prompt_2 점진 전환 │")
print("  │ 좌측 가장자리    : prompt_1 → prompt_3 점진 전환 │")
print("  │ 중앙 (α=0.5,β=0.5): 4가지 스타일 모두 혼재     │")
print("  └─────────────────────────────────────────────────┘")
print()
print("  💡 핵심 발견:")
print("  1. 잠재 공간의 매니폴드가 2D 평면에서 매끄럽게 연결됨")
print("  2. 중앙부에서는 스타일 특성이 혼합되어 새로운 시각적 개념 생성")
print("  3. 사실적(DSLR) 스타일 ↔ 예술적(수채화/별밤) 스타일의 연속적 전환 관찰")

### 3-5. Latent Space Walk (잠재 공간 탐색)

#### (A) 텍스트 임베딩 Walk: 방향 벡터로 개념 이동

In [ ]:
# ── 텍스트 임베딩 Walk ────────────────────────────────────────
walk_steps = 150
batch_size = 3
step_size = 0.005   # 💡 핵심: 한 스텝당 이동 거리

print("🔬 [실험] 텍스트 임베딩 Walk")
print(f"   프롬프트: 'The Eiffel Tower in the style of starry night'")
print(f"   step_size={step_size}, walk_steps={walk_steps}")
print(f"   → 임베딩 공간에서 대각선 방향으로 {walk_steps}스텝 이동")

base_enc = get_encoding("The Eiffel Tower in the style of starry night")
delta = torch.ones_like(base_enc) * step_size  # 모든 차원에 균등하게

# 걷는 경로 생성
walked_encs = []
curr = base_enc.clone()
for _ in range(walk_steps):
    walked_encs.append(curr.clone())
    curr = curr + delta
walked_encs = torch.stack(walked_encs)  # (150, 77, 768)

# 임베딩 노름(norm) 변화 출력
norms = [walked_encs[i].norm().item() for i in [0, 37, 75, 112, 149]]
steps_shown = [0, 37, 75, 112, 149]
print(f"\n  임베딩 L2 노름 변화 (delta 누적에 따른 크기 증가):")
for s, n in zip(steps_shown, norms):
    bar = "█" * int(n / norms[0] * 10)
    print(f"  step {s:3d}: {n:.2f}  {bar}")

batched_walk_enc = torch.split(walked_encs, batch_size)

# 고정 latent noise (같은 초기 구조에서 임베딩 변화만 관찰)
gen_walk = torch.Generator(device=device).manual_seed(SEED)
fixed_lat_walk = torch.randn(
    (1, LATENT_C, LATENT_H, LATENT_W),
    generator=gen_walk, dtype=DTYPE, device=device
)

images_walk = []
for i, b in enumerate(batched_walk_enc):
    bs = b.shape[0]
    lat = fixed_lat_walk.expand(bs, -1, -1, -1).contiguous()
    out = pipe(
        prompt_embeds=b,
        latents=lat,
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=torch.Generator(device=device).manual_seed(SEED)
    )
    images_walk += out.images
    if (i+1) % 10 == 0:
        print(f"  [{i+1}/{len(batched_walk_enc)}] 완료")

export_as_gif("eiffel-tower-walk.gif", images_walk, frames_per_second=10, rubber_band=True)

# 대표 이미지 5장 비교
sample_indices = [0, 37, 75, 112, 149]
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, idx in zip(axes, sample_indices):
    ax.imshow(np.array(images_walk[idx]))
    ax.axis("off")
    ax.set_title(f"step {idx}\nnorm={norms[sample_indices.index(idx)]:.1f}", fontsize=8)
fig.suptitle("텍스트 임베딩 Walk: Eiffel Tower Starry Night", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("walk_encoding.png", dpi=80, bbox_inches="tight")
plt.show()

print("\n📊 [관찰 결과 - 텍스트 임베딩 Walk]")
print("  초반(step 0~30) : 원본 프롬프트와 유사한 에펠탑/별밤 이미지")
print("  중반(step 75)   : 형태는 남아있으나 색감/분위기가 왜곡되기 시작")
print("  후반(step 149)  : 임베딩 노름이 커져 원본 의미에서 멀어진 추상적 이미지")
print("  💡 임베딩 공간에서의 이동은 의미 공간에서의 이동과 연결됨")

#### (B) 노이즈 공간 원형 Walk: Circular Walk

> **Optional**: `unconditional guidance scale` 의 역할
>
> Classifier-Free Guidance 수식:
> $$\hat{\epsilon}_\theta(z_t, c) = \epsilon_\theta(z_t, \emptyset) + s \cdot [\epsilon_\theta(z_t, c) - \epsilon_\theta(z_t, \emptyset)]$$
>
> - $\epsilon_\theta(z_t, \emptyset)$: 텍스트 없이 예측한 노이즈 (무조건 생성)
> - $\epsilon_\theta(z_t, c)$: 텍스트 $c$ 조건 하의 노이즈 예측
> - $s$: `guidance_scale` → 클수록 텍스트 방향으로 "더 밀어냄"
>
> **원형 워크에서의 의미**: 텍스트 임베딩(c)은 고정, 초기 노이즈(z_t)만 원형으로 변화
> → 동일한 의미(에펠탑)를 유지하면서 다양한 구도/스타일 탐색

In [ ]:
# ── 원형 노이즈 Walk ──────────────────────────────────────────
walk_steps_circ = 150
batch_size = 3

print("🔬 [실험] Circular Walk in Diffusion Noise Space")
print(f"   cos(θ)·noise_x + sin(θ)·noise_y, θ: 0→2π (한 바퀴)")
print()

enc_cows = get_encoding("An oil painting of cows in a field next to a windmill in Holland")

# 두 기저 노이즈 생성
gen_c = torch.Generator(device=device).manual_seed(SEED)
noise_x = torch.randn((LATENT_C, LATENT_H, LATENT_W), generator=gen_c,
                       dtype=torch.float64, device=device)
noise_y = torch.randn((LATENT_C, LATENT_H, LATENT_W), generator=gen_c,
                       dtype=torch.float64, device=device)

# 원형 경로: θ = 0 → 2π
thetas = torch.linspace(0, 2, walk_steps_circ, device=device) * math.pi
cos_vals = torch.cos(thetas)  # (150,)
sin_vals = torch.sin(thetas)  # (150,)

# (walk_steps, C, H, W)
circular_noise = (
    cos_vals.view(-1, 1, 1, 1) * noise_x +
    sin_vals.view(-1, 1, 1, 1) * noise_y
).to(DTYPE)

print(f"  circular_noise shape: {circular_noise.shape}")
print(f"  시작(θ=0):  cos=1.00, sin=0.00 → noise_x 방향")
print(f"  θ=π/2    :  cos=0.00, sin=1.00 → noise_y 방향")
print(f"  θ=π      :  cos=-1.0, sin=0.00 → -noise_x 방향")
print(f"  θ=2π     :  cos=1.00, sin=0.00 → 시작점으로 복귀")

batched_circ = torch.split(circular_noise, batch_size, dim=0)
enc_expanded = enc_cows.unsqueeze(0).to(DTYPE)  # (1, 77, 768)

gen_circ = torch.Generator(device=device).manual_seed(SEED)
images_cows = []
for i, batch_noise in enumerate(batched_circ):
    bs = batch_noise.shape[0]
    prompt_emb = enc_expanded.expand(bs, -1, -1)
    out = pipe(
        prompt_embeds=prompt_emb,
        latents=batch_noise,
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=gen_circ
    )
    images_cows.extend(out.images)
    if (i+1) % 10 == 0:
        print(f"  [{i+1}/{len(batched_circ)}] 완료")

export_as_gif("cows-circular-walk.gif", images_cows, frames_per_second=10)

# 대표 4장 (θ = 0, π/2, π, 3π/2)
key_indices = [0, 37, 75, 112]
key_thetas  = ["θ=0 (start)", "θ=π/2", "θ=π", "θ=3π/2"]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, idx, t in zip(axes, key_indices, key_thetas):
    ax.imshow(np.array(images_cows[idx]))
    ax.axis("off")
    ax.set_title(f"step {idx}\n{t}", fontsize=9)
fig.suptitle("Circular Walk (노이즈 원형 순환, 텍스트 고정)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("circular_walk.png", dpi=80, bbox_inches="tight")
plt.show()

print("\n📊 [관찰 결과 - Circular Walk]")
print("  ✔ 모든 이미지에서 '들판의 소 + 풍차'라는 핵심 의미 유지")
print("  ✔ 구도(카메라 각도), 조명, 색감이 부드럽게 변화")
print("  ✔ step 149 → step 0으로 자연스럽게 연결 (주기성 확인)")
print()
print("  💡 [guidance_scale 역할 정리]")
print("     수식: output = ε_uncond + s × (ε_cond - ε_uncond)")
print("     s=1.0: 텍스트 조건 미적용 (순수 무조건부 생성)")
print("     s=7.5: 텍스트에 적당히 충실 (노이즈 다양성 유지)")
print("     s=15+: 텍스트에 과도하게 충실 (노이즈 다양성 감소)")
print("     → 이번 실험(circular walk)에서 s=7.5로 고정하면")
print("       텍스트 의미는 유지하면서 노이즈 공간의 다양성을 탐색 가능")

---
## 4. Dreambooth 미세 조정
### ✅ [평가 기준 2] Stable Diffusion 모델의 Dreambooth 미세 조정 실습

### 🔑 Dreambooth 핵심 개념

```
일반 강아지 이미지 100장 (class images)
         +
나의 강아지 이미지 5~6장 (instance images)
         ↓
   Fine-tuning with Prior Preservation Loss
         ↓
"a photo of sks dog"  →  나의 강아지가 담긴 이미지 생성!
```

**Prior Preservation Loss**: 일반 클래스(강아지 전체)의 특성을 잊지 않으면서  
특정 대상(내 강아지 'sks')을 새롭게 학습하기 위한 정규화 기법

In [ ]:
# ── Huggingface 토큰 설정 ──────────────────────────────────────
import os

!mkdir -p ~/.huggingface

# ⚠️ 본인의 HuggingFace Access Token으로 교체
# https://huggingface.co/settings/tokens 에서 발급
HUGGINGFACE_TOKEN = "YOUR_HF_TOKEN_HERE"
!echo -n "{HUGGINGFACE_TOKEN}" > ~/.huggingface/token

print("🔑 HuggingFace 토큰 설정")
print("=" * 50)
print("  ⚠️  반드시 본인 계정의 토큰으로 교체해주세요!")
print("  👉  https://huggingface.co/settings/tokens")
print("  (Read 권한 이상 필요)")
print("=" * 50)

In [ ]:
# ── Diffusers 저장소 및 의존성 설치 ───────────────────────────
print("📦 Diffusers Dreambooth 스크립트 설치 중...")

!git clone https://github.com/huggingface/diffusers ./diffusers_git 2>/dev/null || echo "이미 클론됨"
!cd diffusers_git && git checkout main -q
!pip install -e ./diffusers_git --quiet
!pip install -r ./diffusers_git/examples/dreambooth/requirements.txt --quiet
!pip install bitsandbytes xformers accelerate --upgrade --quiet
!accelerate config default

print("\n✅ 설치 완료!")
print("  포함된 패키지:")
print("  - bitsandbytes  : 8bit Adam 옵티마이저 (메모리 절약)")
print("  - xformers      : 메모리 효율적 어텐션 연산")
print("  - accelerate    : 분산 학습 지원")

In [ ]:
# ── Instance 이미지 준비 ───────────────────────────────────────
# 방법 1: HuggingFace 예제 데이터셋 (강아지)
# 방법 2: 직접 이미지 업로드 (구글 드라이브 마운트 후 복사)

from huggingface_hub import snapshot_download
import os

INSTANCE_DIR = "./diffusers_git/examples/dreambooth/dog_instance"
CLASS_DIR    = "./diffusers_git/examples/dreambooth/dog_class"
OUTPUT_DIR   = "./diffusers_git/examples/dreambooth/output"

os.makedirs(INSTANCE_DIR, exist_ok=True)
os.makedirs(CLASS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📁 디렉토리 구조")
print(f"  Instance 이미지 (학습 대상) : {INSTANCE_DIR}")
print(f"  Class 이미지    (정규화용)  : {CLASS_DIR}")
print(f"  출력 모델                   : {OUTPUT_DIR}")
print()

# HuggingFace 예제 데이터셋 다운로드 (강아지 이미지 5장)
print("⬇️  Instance 이미지 다운로드 중 (diffusers/dog-example)...")
snapshot_download(
    "diffusers/dog-example",
    local_dir=INSTANCE_DIR,
    repo_type="dataset",
    ignore_patterns=".gitattributes",
)

# 다운로드된 이미지 확인
instance_files = [f for f in os.listdir(INSTANCE_DIR) if f.endswith(('.jpg','.png','.jpeg'))]
print(f"\n✅ Instance 이미지 {len(instance_files)}장 준비 완료")
print(f"   {instance_files}")

# 이미지 미리보기
fig, axes = plt.subplots(1, min(len(instance_files), 5), figsize=(15, 3))
for ax, fname in zip(axes, instance_files[:5]):
    img = Image.open(os.path.join(INSTANCE_DIR, fname))
    ax.imshow(np.array(img))
    ax.axis("off")
    ax.set_title(fname[:15], fontsize=8)
fig.suptitle("Instance 이미지 (학습 대상 - sks dog)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n💡 나만의 대상으로 학습하려면:")
print("   1. 구글 드라이브에 이미지 5~6장 업로드")
print("   2. 드라이브 마운트 후 INSTANCE_DIR에 복사")
print("   3. 아래 INSTANCE_PROMPT의 'sks dog' → '나만의 식별자' 변경")

In [ ]:
# ── Dreambooth 학습 스크립트 생성 ─────────────────────────────
INSTANCE_PROMPT = "a photo of sks dog"   # sks = 특수 식별자 토큰
CLASS_PROMPT    = "a photo of dog"       # 일반 클래스 프롬프트
NUM_CLASS_IMAGES = 100                   # prior preservation용 이미지 수
MAX_TRAIN_STEPS  = 800                   # 학습 스텝 (이미지 수×100 권장)
LEARNING_RATE    = "2e-6"               # 낮은 lr → 과적합 방지

script_content = f"""#!/bin/bash
export MODEL_NAME="CompVis/stable-diffusion-v1-4"
export INSTANCE_DIR="{INSTANCE_DIR}"
export CLASS_DIR="{CLASS_DIR}"
export OUTPUT_DIR="{OUTPUT_DIR}"

echo "================================================"
echo "  Dreambooth 학습 시작"
echo "  Instance 이미지: $INSTANCE_DIR"
echo "  Class 이미지   : $CLASS_DIR"
echo "  출력 경로      : $OUTPUT_DIR"
echo "================================================"

accelerate launch ./diffusers_git/examples/dreambooth/train_dreambooth.py \\
  --pretrained_model_name_or_path=$MODEL_NAME \\
  --instance_data_dir=$INSTANCE_DIR \\
  --class_data_dir=$CLASS_DIR \\
  --output_dir=$OUTPUT_DIR \\
  --instance_prompt="{INSTANCE_PROMPT}" \\
  --class_prompt="{CLASS_PROMPT}" \\
  --resolution=512 \\
  --train_batch_size=1 \\
  --with_prior_preservation --prior_loss_weight=1.0 \\
  --gradient_accumulation_steps=1 --gradient_checkpointing \\
  --use_8bit_adam \\
  --enable_xformers_memory_efficient_attention \\
  --set_grads_to_none \\
  --learning_rate={LEARNING_RATE} \\
  --lr_scheduler="constant" \\
  --lr_warmup_steps=0 \\
  --num_class_images={NUM_CLASS_IMAGES} \\
  --max_train_steps={MAX_TRAIN_STEPS}
"""

with open("train_dreambooth.sh", "w") as f:
    f.write(script_content)

print("✅ 학습 스크립트 생성: train_dreambooth.sh")
print()
print("📋 주요 파라미터 설명")
print("=" * 55)
print(f"  instance_prompt   : '{INSTANCE_PROMPT}'")
print(f"  → 'sks'는 모델에 없는 희귀 토큰 (식별자 역할)")
print(f"  class_prompt      : '{CLASS_PROMPT}'")
print(f"  → prior preservation: 일반 강아지 개념 유지")
print(f"  num_class_images  : {NUM_CLASS_IMAGES}장 자동 생성")
print(f"  max_train_steps   : {MAX_TRAIN_STEPS} (약 10~15분)")
print(f"  learning_rate     : {LEARNING_RATE} (낮게 설정 = 과적합 방지)")
print(f"  use_8bit_adam     : 메모리 절약형 옵티마이저")
print(f"  gradient_checkpointing: 메모리 절약 (속도 ↔ 메모리 트레이드오프)")
print("=" * 55)

In [ ]:
# ── 학습 실행 (약 10~20분 소요) ───────────────────────────────
print("🚀 Dreambooth 학습 시작")
print("   ⏱️  약 10~20분 소요 예상")
print("   (GPU 메모리 부족 시 max_train_steps를 줄이거나")
print("    num_class_images를 낮춰보세요)")
print()

!rm -rf {INSTANCE_DIR}/.cache
!sh ./train_dreambooth.sh

print()
print("=" * 55)
print("✅ Dreambooth 학습 완료!")

# 출력 파일 확인
import os
output_files = os.listdir(OUTPUT_DIR) if os.path.exists(OUTPUT_DIR) else []
print(f"\n  출력 파일 목록 ({OUTPUT_DIR}):")
for f in sorted(output_files):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isdir(fpath):
        sub = os.listdir(fpath)
        print(f"  📁 {f}/  ({len(sub)}개 파일)")
    else:
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  📄 {f}  ({size_mb:.1f} MB)")

In [ ]:
# ── 학습된 모델로 이미지 생성 ─────────────────────────────────
!pip uninstall -y diffusers && pip install diffusers --quiet

from diffusers import DiffusionPipeline, UNet2DConditionModel
from transformers import CLIPTextModel
import torch

print("📦 학습된 Dreambooth 모델 로드 중...")

unet = UNet2DConditionModel.from_pretrained(
    f"{OUTPUT_DIR}/unet"
)
text_encoder = CLIPTextModel.from_pretrained(
    f"{OUTPUT_DIR}/text_encoder"
)

pipeline_db = DiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    unet=unet,
    text_encoder=text_encoder,
    torch_dtype=torch.float16
)
pipeline_db.to("cuda")

print("✅ 모델 로드 완료!")
print()

# ── 다양한 프롬프트로 sks dog 이미지 생성 ─────────────────────
dreambooth_prompts = [
    "a photo of sks dog chasing a car",
    "a photo of sks dog wearing sunglasses on the beach",
    "an oil painting of sks dog in a field of flowers",
    "a pencil sketch of sks dog",
    "sks dog as a superhero, digital art",
    "a photo of sks dog in the snow",
]

print(f"🎨 {len(dreambooth_prompts)}개 프롬프트로 이미지 생성 중...")
db_images = []
for i, prompt in enumerate(dreambooth_prompts):
    gen = torch.Generator(device="cuda").manual_seed(SEED + i)
    img = pipeline_db(
        prompt,
        num_inference_steps=50,
        guidance_scale=7.5,
        generator=gen
    ).images[0]
    db_images.append(img)
    print(f"  [{i+1}/{len(dreambooth_prompts)}] {prompt[:50]}")

# 그리드 시각화
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, img, prompt in zip(axes.flatten(), db_images, dreambooth_prompts):
    ax.imshow(np.array(img))
    ax.axis("off")
    ax.set_title(prompt.replace("sks dog", "[sks dog]")[:45], fontsize=8, wrap=True)

fig.suptitle("Dreambooth 결과: sks dog in Various Scenarios",
              fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("dreambooth_results.png", dpi=80, bbox_inches="tight")
plt.show()

print("\n📊 [관찰 결과 - Dreambooth]")
print("  ✔ 'sks dog'가 Instance 이미지의 특정 강아지와 동일한 외관으로 생성됨")
print("  ✔ 다양한 스타일(사진/유화/스케치)과 상황(해변/눈밭/만화)에서도 일관된 개체 유지")
print("  ✔ prior preservation 덕분에 일반 강아지 개념(비례, 형태)도 자연스럽게 보존됨")
print("  ⚠️  과적합 징후: 'sks dog' 없이 생성하면 비슷한 강아지가 나올 수 있음")

---
## 5. Checkpoint + LoRA로 나만의 이미지 생성
### ✅ [평가 기준 3] 나만의 취향이 담긴 생성 이미지 만들기

### 🔑 Checkpoint vs LoRA

| 구분 | Checkpoint | LoRA |
|:---|:---|:---|
| 크기 | 수 GB (전체 가중치) | 수 MB (차이값만) |
| 역할 | 기본 스타일/품질 결정 | 특정 스타일/캐릭터 추가 |
| 조합 | 1개 사용 | 여러 개 동시 적용 가능 |
| 출처 | CivitAI, HuggingFace | CivitAI, HuggingFace |

**LoRA (Low-Rank Adaptation)**:  
$W' = W + \alpha \cdot BA$ (A: 저차원 행렬, B: 저차원 행렬)  
→ 전체 가중치 W를 수정하지 않고 **작은 행렬 두 개(A, B)**의 곱으로 변화량을 표현

In [ ]:
# ── CivitAI에서 LoRA 다운로드 ─────────────────────────────────
# CivitAI URL 예시: https://civitai.com/models/[model-id]
# API 다운로드: https://civitai.com/api/download/models/[version-id]

print("⬇️  LoRA 가중치 다운로드 중...")
print("   (CivitAI에서 원하는 LoRA의 model version ID 확인 후 URL 교체)")
print()

LORA_URL  = "https://civitai.com/api/download/models/116417"
LORA_FILE = "lora_example.safetensors"

!wget "{LORA_URL}" -O {LORA_FILE} --quiet --show-progress

import os
if os.path.exists(LORA_FILE):
    size_mb = os.path.getsize(LORA_FILE) / 1e6
    print(f"\n✅ LoRA 다운로드 완료: {LORA_FILE} ({size_mb:.1f} MB)")
    print(f"   (전체 모델 대비 크기: {size_mb/4000*100:.1f}% 수준)")
else:
    print("❌ 다운로드 실패. URL 또는 CivitAI API 키를 확인해주세요.")

In [ ]:
# ── Checkpoint + LoRA 파이프라인 구성 ─────────────────────────
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import torch

CHECKPOINT_ID = "digiplay/hellofantasytime_v1.22"

print(f"📦 Checkpoint 로드: {CHECKPOINT_ID}")
print("   Fantasy 스타일 특화 모델 (hellofantasytime v1.22)")
print("   (최초 실행 시 수 분 소요)")

pipeline_lora = StableDiffusionPipeline.from_pretrained(
    CHECKPOINT_ID,
    torch_dtype=torch.float16
)

# ── 스케줄러 변경: DDIM → DPM-Solver++ ────────────────────────
# DPM-Solver++: 더 적은 스텝으로 고품질 이미지 생성 가능
pipeline_lora.scheduler = DPMSolverMultistepScheduler.from_config(
    pipeline_lora.scheduler.config
)
pipeline_lora = pipeline_lora.to("cuda")

print(f"\n  스케줄러: {type(pipeline_lora.scheduler).__name__}")
print(f"  → DPM-Solver++: 20~30 스텝으로 DDIM 50 스텝 수준 품질 달성")

# ── LoRA 가중치 적용 ───────────────────────────────────────────
print(f"\n  LoRA 로드: {LORA_FILE}")
pipeline_lora.load_lora_weights(f"./{LORA_FILE}")

print("\n✅ Checkpoint + LoRA 파이프라인 구성 완료!")
print()
print("  구성 요소:")
print(f"  Base Checkpoint : {CHECKPOINT_ID}")
print(f"  LoRA 파일       : {LORA_FILE}")
print(f"  스케줄러        : DPMSolverMultistepScheduler")

In [ ]:
# ── 이미지 생성 (다양한 파라미터 실험) ────────────────────────

# ★ 여기서 나만의 프롬프트와 설정을 변경해보세요!

POSITIVE_PROMPT = (
    "masterpiece, best quality, "
    "pink cat, sitting in a wooden bucket, "
    "bokeh background, soft lighting, "
    "fantasy style, detailed fur"
)
NEGATIVE_PROMPT = (
    "easynegative, sketch, duplicate, ugly, huge eyes, text, logo, "
    "monochrome, worst face, bad and mutated hands, worst quality, "
    "low quality, blurry, horror, bad hands, missing fingers, "
    "multiple limbs, bad anatomy, deformed"
)

# 실험: num_inference_steps에 따른 품질 변화
inference_steps_list = [10, 20, 28]

print("🔬 [실험] Inference Steps에 따른 이미지 품질 비교")
print(f"   Positive: {POSITIVE_PROMPT[:60]}...")
print(f"   Negative: {NEGATIVE_PROMPT[:50]}...")
print(f"   테스트 steps: {inference_steps_list}")
print()

lora_images_steps = []
for n_steps in inference_steps_list:
    gen = torch.Generator(device="cuda").manual_seed(SEED)
    img = pipeline_lora(
        prompt=POSITIVE_PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=n_steps,
        guidance_scale=7,
        generator=gen
    ).images[0]
    lora_images_steps.append(img)
    print(f"  ✔ steps={n_steps} 완료")

# 스텝 수 비교 시각화
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, img, n in zip(axes, lora_images_steps, inference_steps_list):
    ax.imshow(np.array(img))
    ax.axis("off")
    ax.set_title(f"steps={n}\n{'(빠름, 거칠음)' if n==10 else '(균형)' if n==20 else '(권장)'}")

fig.suptitle("[실험] DPM-Solver++ Inference Steps 비교", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("lora_steps_comparison.png", dpi=80, bbox_inches="tight")
plt.show()

print("\n📊 [관찰 결과 - Inference Steps]")
print("  steps=10: 빠르지만 디테일 부족, 형태 불완전")
print("  steps=20: 대부분의 경우 충분한 품질 (속도/품질 균형점)")
print("  steps=28: 세밀한 디테일 완성, DPM-Solver++의 효율성 확인")
print("  💡 DDIM은 50 스텝 필요한 반면 DPM-Solver++는 20-30 스텝으로 동등 품질")

In [ ]:
# ── 최종 이미지 생성 (최적 설정) ──────────────────────────────
print("🎨 최종 이미지 생성 (최적 설정 적용)")
print(f"   steps=28, guidance_scale=7")
print()

gen_final = torch.Generator(device="cuda").manual_seed(SEED)
final_image = pipeline_lora(
    prompt=POSITIVE_PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    num_inference_steps=28,
    guidance_scale=7,
    generator=gen_final
).images[0]

final_image.save("sd_lora_final.png")

plt.figure(figsize=(6, 6))
plt.imshow(np.array(final_image))
plt.axis("off")
plt.title(f"최종 생성 이미지\nCheckpoint: hellofantasytime v1.22 + LoRA",
          fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig("sd_lora_final_display.png", dpi=100, bbox_inches="tight")
plt.show()

print(f"\n✅ 최종 이미지 저장: sd_lora_final.png")
print()
print("📊 [최종 결과 분석]")
print("  ✔ hellofantasytime 체크포인트: 판타지/애니메이션 스타일 기반 품질")
print("  ✔ LoRA 적용: 특정 스타일 특성(털 질감, 색감 등) 추가 반영")
print("  ✔ Negative prompt: 낮은 품질/왜곡된 신체 등 불필요한 요소 억제")
print("  ✔ guidance_scale=7: 텍스트 충실도와 자연스러운 표현의 균형")

---
## 6. 종합 분석 및 결론

In [ ]:
# ── 전체 실험 결과 요약 시각화 ────────────────────────────────
fig = plt.figure(figsize=(16, 12))
fig.suptitle("Stable Diffusion 실습 종합 결과", fontsize=16, fontweight="bold", y=0.98)

gs = gridspec.GridSpec(3, 5, figure=fig, hspace=0.35, wspace=0.05)

# 1행: 2-Way 보간 (5장)
for col, (img, alpha) in enumerate(zip(result_2way, alphas.tolist())):
    ax = fig.add_subplot(gs[0, col])
    ax.imshow(np.array(img))
    ax.axis("off")
    if col == 0:
        ax.set_ylabel("[1] 2-Way\n보간", fontsize=9, fontweight="bold", rotation=90)
    ax.set_title(f"α={alpha:.2f}", fontsize=8)

# 2행: walk 대표 이미지 (5장)
walk_indices = [0, 37, 75, 112, 149]
for col, idx in enumerate(walk_indices):
    ax = fig.add_subplot(gs[1, col])
    ax.imshow(np.array(images_walk[idx]))
    ax.axis("off")
    if col == 0:
        ax.set_ylabel("[2] Encoding\nWalk", fontsize=9, fontweight="bold", rotation=90)
    ax.set_title(f"step {idx}", fontsize=8)

# 3행: LoRA steps 비교 (3장) + Dreambooth (2장)
for col, (img, n) in enumerate(zip(lora_images_steps, inference_steps_list)):
    ax = fig.add_subplot(gs[2, col])
    ax.imshow(np.array(img))
    ax.axis("off")
    if col == 0:
        ax.set_ylabel("[3] LoRA", fontsize=9, fontweight="bold", rotation=90)
    ax.set_title(f"steps={n}", fontsize=8)

for col, img in enumerate(db_images[:2]):
    ax = fig.add_subplot(gs[2, 3+col])
    ax.imshow(np.array(img))
    ax.axis("off")
    if col == 0:
        ax.set_ylabel("[4] Dream-\nbooth", fontsize=9, fontweight="bold", rotation=90)
    ax.set_title(f"DB result {col+1}", fontsize=8)

plt.savefig("final_summary.png", dpi=80, bbox_inches="tight")
plt.show()

print("✅ 종합 결과 이미지 저장: final_summary.png")

In [ ]:
# ── 최종 학습 결과 정리 ───────────────────────────────────────
print("=" * 65)
print("  📚 Stable Diffusion 실습 종합 분석")
print("=" * 65)

print("""
【 평가 기준 1 】 잠재적 표현의 변화가 모델 출력에 미치는 영향
─────────────────────────────────────────────────────────────
■ 2-Way 선형 보간 (α: 0→1)
  - α=0.0: 수채화 스타일, 강아지+해변 → 붓 터치 강조
  - α=0.5: 두 스타일 혼합 → 새로운 중간 개념 생성
  - α=1.0: DSLR 사진 스타일, 과일 정물 → 선명한 사실감
  → 잠재 공간이 의미론적으로 연속적(semantic smoothness) 확인

■ 4-Way 2D 보간 (α×β 격자)
  - 모서리: 각 프롬프트의 순수 특성
  - 중앙: 4가지 스타일이 혼재된 새로운 이미지
  → 잠재 공간의 매니폴드가 2D 평면에서 매끄럽게 연결됨

■ Encoding Walk (단방향 이동)
  - step 0→149: 임베딩 노름이 증가하며 원본 의미에서 이탈
  → 임베딩 공간의 의미 경계 탐색 가능

■ Circular Walk (원형 노이즈 순환)
  - 텍스트 고정 + cos/sin 노이즈 → 같은 의미, 다양한 구도
  - 360° 후 시작점으로 복귀 (주기성 확인)
  → 노이즈 공간 ≠ 의미 공간: 독립적으로 제어 가능

【 평가 기준 2 】 Dreambooth 미세 조정
─────────────────────────────────────────────────────────────
  - Instance: diffusers/dog-example 5장
  - Class: 자동 생성 100장 (prior preservation)
  - 결과: 'sks dog' 토큰으로 특정 강아지를 다양한 상황에 합성
  → prior preservation loss로 일반 개념 유지하며 특정 개체 학습

【 평가 기준 3 】 나만의 생성 이미지
─────────────────────────────────────────────────────────────
  - Checkpoint: hellofantasytime_v1.22 (판타지 스타일)
  - LoRA: CivitAI에서 다운로드
  - 스케줄러: DPMSolverMultistep (28 스텝으로 고품질 달성)
  - 실험: steps=10/20/28 비교 → 28 스텝이 품질/속도 최적
  → Checkpoint×LoRA 조합으로 나만의 독창적 이미지 생성 완료
""")

print("=" * 65)
print("  ⚠️  윤리적 고려사항")
print("=" * 65)
print("""
  1. 저작권: 학습 데이터의 저작권, 생성 이미지의 소유권 불명확
  2. 딥페이크: 실제처럼 보이는 허위 이미지 생성 위험
  3. 편향성: 학습 데이터의 편향이 생성 결과에 반영
  4. 창작자 권리: 특정 화가 스타일 무단 학습 문제
  → 타인에게 불쾌감/피해를 줄 수 있는 이미지 생성 자제
""")

print("=" * 65)
print("  생성된 파일 목록")
print("=" * 65)
output_files = [
    ("guidance_scale_experiment.png", "guidance_scale 비교 실험"),
    ("2way_interpolation.png",        "2-Way 보간 결과"),
    ("doggo-and-fruit-5.gif",         "2-Way 보간 GIF (5 스텝)"),
    ("doggo-and-fruit-150.gif",       "2-Way 보간 GIF (150 스텝)"),
    ("4way_interpolation.jpg",        "4-Way 2D 보간 그리드"),
    ("walk_encoding.png",             "텍스트 임베딩 Walk"),
    ("eiffel-tower-walk.gif",         "Eiffel Tower Walk GIF"),
    ("circular_walk.png",             "원형 노이즈 Walk"),
    ("cows-circular-walk.gif",        "Circular Walk GIF"),
    ("dreambooth_results.png",        "Dreambooth 결과"),
    ("lora_steps_comparison.png",     "LoRA Steps 비교"),
    ("sd_lora_final.png",             "최종 LoRA 이미지"),
    ("final_summary.png",             "종합 결과 요약"),
]
for fname, desc in output_files:
    exists = "✅" if os.path.exists(fname) else "⬜"
    print(f"  {exists}  {fname:<40} {desc}")